## Setup

In [1]:
import json

#get params
with open("./params.json", mode = "r", encoding = "utf-8") as f:
    data = json.load(f)
    seed_vals = data["seed_vals"]
    ensemble_root_path = data["ensemble_root_path"]
    dataset_path_test_stream_windowed = data["dataset_path"]["test"]["stream"]["windowed"]
    dataset_path_test_stream_full = data["dataset_path"]["test"]["stream"]["full"]
    dataset_path_test_reversal_windowed = data["dataset_path"]["test"]["reversal"]["windowed"]
    dataset_path_test_reversal_full = data["dataset_path"]["test"]["reversal"]["full"]
    stats_path = data["stats_path"]
    num_single_sample_timesteps = data["num_single_sample_timesteps"]
    input_window_length = data["input_window_length"]
    label_window_length = data["label_window_length"]
    valid_length = input_window_length + label_window_length
    input_features = data["input_features"]
    label_features = data["label_features"]
    extra_features = data["extra_features"]
    num_label_features = len(label_features)
    window_stride = data["window_stride"]
    num_datapoints_per_timeseries = len(
        list(
            range(
                0, num_single_sample_timesteps - (input_window_length + label_window_length) + 1, window_stride
            )
        )
    )
    num_datapoints_per_timeseries_stream = len(
        list(
            range(
                0, 50000 - (input_window_length + label_window_length) + 1, window_stride
            )
        )
    )
    # num_datapoints_per_timeseries = 1 + (num_single_sample_timesteps - (input_window_length + label_window_length) + 1) // window_stride

    seed_val = 0

    # Use for kdeplot inference
    batch_size = data["batch_size"]

In [2]:
import torch
import random
import numpy as np
import pandas as pd
from scipy import stats as stats_module

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(seed_val)
random.seed(seed_val)
np.random.seed(seed_val)

In [3]:
# from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import ipywidgets as widgets
import polars as pl
import ast

from utils.pipeline.Model import TimeSeriesHuggingFaceTransformer
# from utils.pipeline.Data import get_mean_std_respected_temporal, WindowedIterableDataset, read_stats
from utils.pipeline.Data import InferenceAnalysisDataset, WindowedDataset
from utils.pipeline.Run import forward_pass

In [4]:
### REVERSALS ###

stats = pl.read_csv(stats_path)

df_test = InferenceAnalysisDataset(
    windowed_dataset_path = dataset_path_test_reversal_windowed,
    full_dataset_path = dataset_path_test_reversal_full,
    index_full_update_len = num_datapoints_per_timeseries
)

In [91]:
### Global Params
num_timeseries = 803
first_window = 69
num_windows = 11

#model paths|
model_paths = [
    f"{ensemble_root_path}/22/T5-{input_window_length}-{label_window_length}-{window_stride}-{val}.pt" for val in seed_vals
]

## Computations

### Reversal Probabilities

In [37]:
#Get the number of models that predict a reversal (reversal probability) for windows 69-79 for all timeseries

chunk_size = 128

#create dataframes to hold the reversal probabilities
original_reversal_probs = pd.DataFrame(
    index=range(num_timeseries),
    columns=[str(j) for j in range(first_window, 80)],
)

normal_reversal_probs = pd.DataFrame(
    index=range(num_timeseries),
    columns=[str(j) for j in range(first_window, 80)],
)

good_reversal_probs = pd.DataFrame(
    index=range(num_timeseries),
    columns=[str(j) for j in range(first_window, 80)],
)


num_windows = 80 - first_window

#build a grid of window-timeseries pairs
i_grid, j_grid = np.meshgrid(
    np.arange(num_timeseries), np.arange(first_window, 80), indexing="ij"
)
i_flat = i_grid.flatten()
j_flat = j_grid.flatten()
data_indices = i_flat * num_datapoints_per_timeseries + j_flat
N = data_indices.shape[0]

#get an array of windows; each index is a specific window-timeseries pair
windows_x = np.asarray(df_test.inputs_windowed[data_indices]) 
windows_y = np.asarray(df_test.labels_windowed[data_indices])

#similarly, get arrays for label features and extra features
index_full = data_indices // df_test.index_full_update_len
labels_full_batch = np.asarray(df_test.labels_full[index_full])
extras_full_batch = np.asarray(df_test.extras_full[index_full])

#make the numpy arrays torch tensors?
batch_x = torch.from_numpy(windows_x).float().to(device)
batch_y = torch.from_numpy(windows_y).float().to(device) 
x_labels = torch.from_numpy(labels_full_batch).float()
extra_full = torch.from_numpy(extras_full_batch).float()

#empty tensor/array for predictions/losses
all_preds = torch.zeros((len(model_paths), N) + batch_y.shape[1:])
#all_attns = torch.zeros((len(model_paths), N, label_window_length, input_window_length))
losses = np.zeros((len(model_paths), N))
all_residuals = torch.zeros((len(model_paths), N) + batch_y.shape[1:])

#L1 loss as our loss function
criterion = torch.nn.L1Loss(reduction="none")

#for every model in the ensemble
for model_idx, path in enumerate(model_paths):

    #load the model
    model = torch.load(path, map_location=device, weights_only=False).to(device)
    model.eval()

    #needed because we're only doing forward passes; no backpropagation
    with torch.no_grad():
        #loop through all data in chunks
        for start in range(0, N, chunk_size):
            end = min(start + chunk_size, N)

            #forward pass of the model with the specific chunk of data
            outputs = forward_pass(
                model=model,
                batch_x=batch_x[start:end],
                device=device,
                extract_attention=True
            )

            #get the predictions
            preds = outputs.logits

            #residuals
            residuals = batch_y[start:end] - preds

            #calculate loss for each model and timeseries?
            per_sample_loss = criterion(preds, batch_y[start:end]).mean(dim=(1, 2))

            #put the predictions/losses into the tensor/array
            all_preds[model_idx, start:end] = preds.cpu()
            all_residuals[model_idx, start:end] = residuals.cpu()
            #all_attns[model_idx, start:end] = model.get_average_attention_values()  # will be silently wrong if uncommented
            losses[model_idx, start:end] = per_sample_loss.cpu().numpy()

    del model
    torch.cuda.empty_cache()

#get what index u_list is in the list of label features
#at time of writing, u_idx is 2 because label_features is ("b_e", "b_plus", "u_list")
u_idx = label_features.index("u_list")

#get number of models
num_models = len(model_paths)

#get predictions for U from the first and last timestep in a timeseries-window pair
first_timesteps = all_preds[:, :, 0, u_idx]
last_timesteps = all_preds[:, :, -1, u_idx]

##check whether there was a reserval event based on different criteria
#original criterion used. First timestep positive and last timestep negative. Misses some predicted reservals when last U in window is positive, but first prediction is negative
pred_reversal_original = (torch.sign(first_timesteps) != torch.sign(last_timesteps))
pred_reversal_original = pred_reversal_original.reshape(num_models, num_timeseries, num_windows)

#new criterion used. Last timestep is negative. Only works for reversal event data
pred_reversal_normal = (last_timesteps < 0)
pred_reversal_normal = pred_reversal_normal.reshape(num_models, num_timeseries, num_windows)

#criterion used to asses whether reversal was predicted at roughly the correst time. Does not count as a predicted reversal if prediction crosses u=0 too early (too much error)
pred_reversal_good = ((last_timesteps < 0) & (torch.from_numpy(losses) < 1))
pred_reversal_good = pred_reversal_good.reshape(num_models, num_timeseries, num_windows)


##check whether there was a reserval event based on different criteria
#original criterion used. First timestep positive and last timestep negative. Misses some predicted reservals when last U in window is positive, but first prediction is negative
count_pred_reversal_original = (torch.sign(first_timesteps) != torch.sign(last_timesteps)).sum(dim=0).numpy()
#new criterion used. Last timestep is negative. Only works for reversal event data
count_pred_reversal_normal = (last_timesteps.numpy() < 0).sum(axis=0)
#criterion used to asses whether reversal was predicted at roughly the correst time. Does not count as a predicted reversal if prediction crosses u=0 too early (too much error)
count_pred_reversal_good = ((last_timesteps.numpy() < 0) & (losses < 1)).sum(axis=0)     



#get probabilities
original_reversal_prob = (count_pred_reversal_original/num_models)*100 
normal_reversal_prob = (count_pred_reversal_normal/num_models)*100 
good_reversal_prob = (count_pred_reversal_good/num_models)*100

#Reshape 1d arrays/tensors into priorly created dataframes
original_reversal_probs.iloc[:, :] = original_reversal_prob.reshape(num_timeseries, num_windows)
normal_reversal_probs.iloc[:, :] = normal_reversal_prob.reshape(num_timeseries, num_windows)
good_reversal_probs.iloc[:, :] = good_reversal_prob.reshape(num_timeseries, num_windows)

#reshape preds (10, 803, 11, 100, 3)
preds = all_preds.reshape(num_models, num_timeseries, num_windows, 50, len(label_features))
residuals = all_residuals.reshape(num_models, num_timeseries, num_windows, 50, len(label_features))
features = batch_x.reshape(num_timeseries, num_windows, 100, len(label_features))

In [38]:
#save reversal probabilities dataframes as csv files so we don't need to run the above code every time (takes ~10 minutes on my machine)
original_reversal_probs.to_csv('./analysis/reversal_probs/original_reversal_probs.csv', index=False)
normal_reversal_probs.to_csv('./analysis/reversal_probs/normal_reversal_probs.csv', index=False)
good_reversal_probs.to_csv('./analysis/reversal_probs/good_reversal_probs.csv', index=False)

#save reversal predictions by model
torch.save(pred_reversal_original, './analysis/reversal_prediction_by_model/pred_reversal_original.pt')
torch.save(pred_reversal_normal, './analysis/reversal_prediction_by_model/pred_reversal_normal.pt')
torch.save(pred_reversal_good, './analysis/reversal_prediction_by_model/pred_reversal_good.pt')

#save predictions and residuals
torch.save(preds, './analysis/full_preds.pt')
torch.save(residuals, './analysis/full_residuals.pt')
torch.save(features, './analysis/full_features.pt')

### Feature Sorting by Reversal Probabilities

In [ ]:
#Run once for each of original, normal, and good
reversal_probs = original_reversal_probs

#empty tensors
features_100 = []
features_high = []
features_mid = []
features_low = []
features_0 = []

#loop through windows
for window_idx in (reversal_probs.columns):

    #current window index as an integer
    selected_window_idx = int(window_idx)

    #temporary tensors
    temp_100 = []
    temp_high = []
    temp_mid = []
    temp_low = []
    temp_0 = []

    #for each timeseries (at the specific window)
    for i in range(num_timeseries):
        #current timeseries
        target_timeseries_idx = i

        #index in the 1d array
        data_idx = target_timeseries_idx * num_datapoints_per_timeseries + selected_window_idx
        window_x, window_y, x_labels, extra_full = df_test[data_idx]

        #grab the reversal probability for the current timeseries and window index
        reversal_prob = reversal_probs[window_idx][target_timeseries_idx]

        #sort based on reversal probabilities and add to temporary tensors
        if(reversal_prob == 100):
            temp_100.append(window_x)
        if(reversal_prob >= 80):
            temp_high.append(window_x)
        if((reversal_prob < 80) & (reversal_prob > 20)):
            temp_mid.append(window_x)
        if(reversal_prob <= 20):
            temp_low.append(window_x)
        if(reversal_prob == 0):
            temp_0.append(window_x)

    #add temporary tensors to actual tensors
    features_100.append(torch.stack(temp_100) if temp_100 else torch.empty(0))
    features_high.append(torch.stack(temp_high) if temp_high else torch.empty(0))
    features_mid.append(torch.stack(temp_mid) if temp_mid else torch.empty(0))
    features_low.append(torch.stack(temp_low) if temp_low else torch.empty(0))
    features_0.append(torch.stack(temp_0) if temp_0 else torch.empty(0))


In [ ]:
#save tensors so we don't need to run the above code every time (takes ~10 minutes on my machine)
torch.save(features_100, './analysis/features_sorted_by_reversal_probs/original_features_100.pt')
torch.save(features_high, './analysis/features_sorted_by_reversal_probs/original_features_high.pt')
torch.save(features_mid, './analysis/features_sorted_by_reversal_probs/original_features_mid.pt')
torch.save(features_low, './analysis/features_sorted_by_reversal_probs/original_features_low.pt')
torch.save(features_0, './analysis/features_sorted_by_reversal_probs/original_features_0.pt')

### Residuals Sorting by Reversal Probabilities

In [ ]:
#Run once for each of original, normal, and good
pred_reversal = original_pred_reversal
num_windows = 11
num_models = 10

#empty tensors
residuals_true = []
residuals_false = []

#loop through windows
for window_idx in range(num_windows):

    #temporary tensors
    temp_resids_true = []
    temp_resids_false = []

    #for each timeseries (at the specific window)
    for model_idx in range(num_models):

        #more temporary tensors
        double_temp_resids_true = []
        double_temp_resids_false = []

        for timeseries_idx in range(num_timeseries):
            if(pred_reversal[model_idx, timeseries_idx, window_idx]):
                double_temp_resids_true.append(residuals[model_idx, timeseries_idx, window_idx, :, :])
            else:
                double_temp_resids_false.append(residuals[model_idx, timeseries_idx, window_idx, :, :])

        #add double temporary tensors to temporary tensors
        temp_resids_true.append(torch.stack(double_temp_resids_true) if double_temp_resids_true else torch.empty(0))
        temp_resids_false.append(torch.stack(double_temp_resids_false) if double_temp_resids_false else torch.empty(0))

    #add temporary tensors to actual tensors
    residuals_true.append(temp_resids_true)
    residuals_false.append(temp_resids_false)



In [ ]:
torch.save(residuals_true, './analysis/residuals_sorted_by_reversal_pred/original_residuals_true.pt')
torch.save(residuals_false, './analysis/residuals_sorted_by_reversal_pred/original_residuals_false.pt')

### Get Stream Data

In [39]:
stats = pl.read_csv(stats_path)

df_test_stream = InferenceAnalysisDataset(
    windowed_dataset_path = dataset_path_test_stream_windowed,
    full_dataset_path = dataset_path_test_stream_full,
    index_full_update_len = num_datapoints_per_timeseries_stream
)

In [40]:
num_streams_wanted = 1000

stream_windows = []
stream_y = []

count = 0
while count < num_streams_wanted:
    rn = random.randint(0, 847534)
    if (rn % 9971 > 4000): #check to see if random number is not at start of stream
        window_x = np.asarray(df_test_stream.inputs_windowed[rn])
        window_y = np.asarray(df_test_stream.labels_windowed[rn])
        if (np.all(window_x[:, 2] > 0) & np.all(window_y[:, 2] > 0)): #check to make sure we are only in the positives for u
            stream_windows.append(torch.from_numpy(window_x))
            stream_y.append(torch.from_numpy(window_y))
            count = count+1

stream_windows = torch.stack(stream_windows)
stream_y = torch.stack(stream_y)

In [41]:
#num windows
num_streams = len(stream_windows)
num_models = len(model_paths)


batch_x = stream_windows.float().to(device)
batch_y = stream_y.float().to(device)

#empty tensor/array for predictions/losses
stream_preds = torch.zeros((len(model_paths), num_streams) + batch_y.shape[1:])
stream_residuals = torch.zeros((len(model_paths), num_streams) + batch_y.shape[1:])

for model_idx, path in enumerate(model_paths):

    #load the model
    model = torch.load(path, map_location=device, weights_only=False).to(device)
    model.eval()

    #needed because we're only doing forward passes; no backpropagation
    with torch.no_grad():


        #forward pass of the model with the specific chunk of data
        outputs = forward_pass(
            model=model,
            batch_x=batch_x,
            device=device,
            extract_attention=False
        )

        #get the predictions
        preds = outputs.logits

        #residuals
        residuals = batch_y - preds

        #put the predictions/losses into the tensor/array
        stream_preds[model_idx, :] = preds.cpu()
        stream_residuals[model_idx, :] = residuals.cpu()

    del model
    torch.cuda.empty_cache()

In [42]:
#save data
torch.save(stream_windows, './analysis/stream/stream_windows.pt')
torch.save(stream_y, './analysis/stream/stream_y.pt')
torch.save(stream_preds, './analysis/stream/stream_preds.pt')
torch.save(stream_residuals, './analysis/stream/stream_residuals.pt')

# Analysis

### Analysis Setup

In [72]:
#read csv files
original_reversal_probs = pd.read_csv('./analysis/reversal_probs/original_reversal_probs.csv')
normal_reversal_probs = pd.read_csv('./analysis/reversal_probs/normal_reversal_probs.csv')
good_reversal_probs = pd.read_csv('./analysis/reversal_probs/good_reversal_probs.csv')

original_pred_reversal = torch.load('./analysis/reversal_prediction_by_model/pred_reversal_original.pt')
normal_pred_reversal = torch.load('./analysis/reversal_prediction_by_model/pred_reversal_normal.pt')
good_pred_reversal = torch.load('./analysis/reversal_prediction_by_model/pred_reversal_good.pt')

original_features_100 = torch.load('./analysis/features_sorted_by_reversal_probs/original_features_100.pt')
original_features_high = torch.load('./analysis/features_sorted_by_reversal_probs/original_features_high.pt')
original_features_mid = torch.load('./analysis/features_sorted_by_reversal_probs/original_features_mid.pt')
original_features_low = torch.load('./analysis/features_sorted_by_reversal_probs/original_features_low.pt')
original_features_0 = torch.load('./analysis/features_sorted_by_reversal_probs/original_features_0.pt')

normal_features_100 = torch.load('./analysis/features_sorted_by_reversal_probs/normal_features_100.pt')
normal_features_high = torch.load('./analysis/features_sorted_by_reversal_probs/normal_features_high.pt')
normal_features_mid = torch.load('./analysis/features_sorted_by_reversal_probs/normal_features_mid.pt')
normal_features_low = torch.load('./analysis/features_sorted_by_reversal_probs/normal_features_low.pt')
normal_features_0 = torch.load('./analysis/features_sorted_by_reversal_probs/normal_features_0.pt')

good_features_100 = torch.load('./analysis/features_sorted_by_reversal_probs/good_features_100.pt')
good_features_high = torch.load('./analysis/features_sorted_by_reversal_probs/good_features_high.pt')
good_features_mid = torch.load('./analysis/features_sorted_by_reversal_probs/good_features_mid.pt')
good_features_low = torch.load('./analysis/features_sorted_by_reversal_probs/good_features_low.pt')
good_features_0 = torch.load('./analysis/features_sorted_by_reversal_probs/good_features_0.pt')

#resids
original_residuals_true = torch.load("./analysis/residuals_sorted_by_reversal_pred/original_residuals_true.pt", weights_only=False)
original_residuals_false = torch.load("./analysis/residuals_sorted_by_reversal_pred/original_residuals_false.pt", weights_only=False)

normal_residuals_true = torch.load("./analysis/residuals_sorted_by_reversal_pred/normal_residuals_true.pt", weights_only=False)
normal_residuals_false = torch.load("./analysis/residuals_sorted_by_reversal_pred/normal_residuals_false.pt", weights_only=False)

good_residuals_true = torch.load("./analysis/residuals_sorted_by_reversal_pred/good_residuals_true.pt", weights_only=False)
good_residuals_false = torch.load("./analysis/residuals_sorted_by_reversal_pred/good_residuals_false.pt", weights_only=False)

#full preds and residuals
full_preds = torch.load('./analysis/full_preds.pt')
full_residuals = torch.load('./analysis/full_residuals.pt')
full_features  = torch.load('./analysis/full_features.pt')
full_y = full_residuals + full_preds
full_y = full_y[0, :, :, :, :]

#stream
stream_windows = torch.load('./analysis/stream/stream_windows.pt')
stream_y = torch.load('./analysis/stream/stream_y.pt')
stream_preds = torch.load('./analysis/stream/stream_preds.pt')
stream_residuals = torch.load('./analysis/stream/stream_residuals.pt')

In [7]:
#Lists
colors = ['#00b7ff', '#d4e000', '#d20b64', '#0037FA', '#7A0000', 'black']
feature_list = ["b_e", "b_plus", "U"]
summary_stat_list = ["Mean", "Median", "Std", "Min", "Max", "Range"]
prediction_type_list = ["Original", "Normal", "Good"]
highlight_type_list = ["100", "High", "Mid", "Low", "0"]

reversal_probs_list = [original_reversal_probs, normal_reversal_probs, good_reversal_probs]
pred_reversal_list = [original_pred_reversal, normal_pred_reversal, good_pred_reversal]

features = [[original_features_100, normal_features_100, good_features_100],
            [original_features_high, normal_features_high, good_features_high],
            [original_features_mid, normal_features_mid, good_features_mid],
            [original_features_low, normal_features_low, good_features_low],
            [original_features_0, normal_features_0, good_features_0]]

residuals = [[original_residuals_true, normal_residuals_true, good_residuals_true],
             [original_residuals_false, normal_residuals_false, good_residuals_false]]

In [8]:
colors_vals = torch.empty(3, num_windows, num_timeseries)

for prediction_type_idx in range(0, 2):
    for window_idx in range(num_windows):
        for timeseries_idx in range(num_timeseries):
            temp_reversal_prob = reversal_probs_list[prediction_type_idx].to_numpy()
            reversal_prob = temp_reversal_prob[timeseries_idx, window_idx]
            if(reversal_prob >= 80):
                colors_vals[prediction_type_idx, window_idx, timeseries_idx] = 0
            elif(reversal_prob <= 20):
                colors_vals[prediction_type_idx, window_idx, timeseries_idx] = 2
            else:
                colors_vals[prediction_type_idx, window_idx, timeseries_idx] = 1

In [9]:
colors_vals_by_mod = torch.empty(2, 10, num_windows, num_timeseries)

for prediction_type_idx in range(0, 2):
    for model_idx in range(len(model_paths)):
        for window_idx in range(num_windows):
            for timeseries_idx in range(num_timeseries):
                pred = pred_reversal_list[prediction_type_idx][model_idx, timeseries_idx, window_idx]
                if(pred):
                    colors_vals_by_mod[prediction_type_idx, model_idx, window_idx, timeseries_idx] = 0
                else:
                    colors_vals_by_mod[prediction_type_idx, model_idx, window_idx, timeseries_idx] = 2

In [10]:
#definition to calculate the relevant summary statistic for a tensor of data based on the window, window subset, feature, and axis
def calculate_summary_stat(tensor, summary_stat, window_idx, window_subset_start, window_subset_end, feature_num, axis, count = 1000):
        #empty array
        return_array = []

        #get relevant data as array
        if(tensor is stream_windows):
                subset = np.asarray(tensor[0:(count-1), window_subset_start:window_subset_end, feature_num])
        elif(tensor is full_features):
                subset = np.asarray(tensor[:, window_idx - first_window, window_subset_start:window_subset_end, feature_num])
        else:
                subset = np.asarray(tensor[window_idx - first_window][:, window_subset_start:window_subset_end, feature_num])

        if (summary_stat == "Mean"):
                return_array =  subset.mean(axis = axis)
        elif (summary_stat == "Median"):
                return_array = np.median(subset, axis = axis)
        elif (summary_stat == "Std"):
                return_array = subset.std(axis = axis)
        elif (summary_stat == "Min"):
                return_array = np.min(subset, axis = axis)
        elif (summary_stat == "Max"):
                return_array = np.max(subset, axis = axis)
        elif (summary_stat == "Range"):
                return_array = np.ptp(subset, axis = axis)

        return return_array

### Define Widget Functions

In [125]:
def make_window_idx(value=0):
    return widgets.IntSlider(
        value=value, min=69, max=79, step=1,
        description="Window Index",
        layout=widgets.Layout(width="500px"),
        style={"description_width": "100px"}
    )

def make_point_idx_50(value=0):
    return widgets.IntSlider(
        value=value, min=0, max=49, step=1,
        description="Point Index",
        layout=widgets.Layout(width="500px"),
        style={"description_width": "100px"}
    )

def make_point_idx_100(value=0):
    return widgets.IntSlider(
        value=value, min=0, max=99, step=1,
        description="Point Index",
        layout=widgets.Layout(width="500px"),
        style={"description_width": "100px"}
    )

def make_highlight_idx(value=0):
    return widgets.IntSlider(
        value=value, min=0, max=1000, step=1,
        description="Highlight Timeseries",
        layout=widgets.Layout(width="500px"),
        style={"description_width": "100px"}
    )

def make_model_idx(value=0):
    return widgets.IntSlider(
        value=value, min=0, max=9, step=1,
        description="Model Index",
        layout=widgets.Layout(width="500px"),
        style={"description_width": "100px"}
    )

def make_count(value=50):
    return widgets.IntSlider(
        value=value, min=0, max=200, step=5,
        description="Count",
        layout=widgets.Layout(width="500px"),
        style={"description_width": "100px"}
    )

def make_target_prob(value=0):
    return widgets.IntSlider(
        value=value, min=0, max=100, step=10,
        description="Target Probability",
        layout=widgets.Layout(width="500px"),
        style={"description_width": "100px"}
    )

def make_window_subset(value=[0,100]):
    return widgets.IntRangeSlider(
        value=value, min=0, max=100, step=1,
        description="Window Subset",
        layout=widgets.Layout(width="500px"),
        style={"description_width": "100px"}
)

def make_feature_name(value="b_plus"):
    return widgets.Dropdown(
        options=feature_list, value=value,
        description="Feature",
        style={"description_width": "100px"}
    )

def make_summary_statistic(value="Mean"):
    return widgets.Dropdown(
        options=summary_stat_list, value = value,
        description = "Summary Statistic",
        style={"description_width": "100px"}
    )

def make_prediction_type(value="Normal"):
    return widgets.Dropdown(
        options = prediction_type_list, value=value,
        description="Prediction Type",
        style={"description_width": "100px"}
    )

def make_highlight_type(value="None"):
    return widgets.Dropdown(
        options=["None", "Reversal", "Stream"], value=value,
        description="Highlight Type",
        style={"description_width": "100px"}
    )

def make_show_lines(value=False):
    return widgets.Checkbox(
        value=value, description="Show Lines",
        indent=True, style={"description_width": "100px"}
    )

def make_show_true(value=True):
    return widgets.Checkbox(
        value=value, description="Show True",
        indent=True, style={"description_width": "100px"}
    )

def make_show_false(value=True):
    return widgets.Checkbox(
        value=value, description="Show False",
        indent=True, style={"description_width": "100px"}
    )

def make_show_high(value=True):
    return widgets.Checkbox(
        value=value, description="Show High",
        indent=True, style={"description_width": "100px"}
    )

def make_show_low(value=True):
    return widgets.Checkbox(
        value=value, description="Show Low",
        indent=True, style={"description_width": "100px"}
    )

def make_show_stream(value=False):
    return widgets.Checkbox(
        value=value, description="Show Stream",
        indent=True, style={"description_width": "100px"}
    )

def make_show_means(value=False):
    return widgets.Checkbox(
        value=value, description="Show Means",
        indent=True, style={"description_width": "100px"}
    )

def make_aggregate(value=False):
    return widgets.Checkbox(
        value=value, description="Aggregate", disabled = False,
        indent=True, style={"description_width": "100px"}
    )

def make_combine_models(value=True):
    return widgets.Checkbox(
        value=value, description="Combine Models", disabled = False,
        indent=True, style={"description_width": "100px"}
    )

def make_combine_count(value=True):
    return widgets.Checkbox(
        value=value, description="Combine Count", disabled = False,
        indent=True, style={"description_width": "100px"}
    )


### Prediction Horizon

In [17]:
#### Define Function####
def bar_of_prediction_horizon(
        prediction_type
    ):
    
    #vars
    prediction_type_number = prediction_type_list.index(prediction_type)
    reversal_probs = reversal_probs_list[prediction_type_number]

    #How many timeseries fit into each reversal_prob category for each window index
    counts_100 = (reversal_probs == 100).sum(axis=0).to_numpy()/np.full(num_windows, num_timeseries/100)
    counts_high = (reversal_probs >= 80).sum(axis=0).to_numpy()/np.full(num_windows, num_timeseries/100)
    counts_low = (reversal_probs <= 20).sum(axis=0).to_numpy()/np.full(num_windows, num_timeseries/100)
    counts_0 = (reversal_probs == 0).sum(axis=0).to_numpy()/np.full(num_windows, num_timeseries/100)
    counts_mid = (np.full(num_windows, 100) - counts_high - counts_low)

    #get the average reversal_prob for the three types of reversal_prob
    averages = [x.mean().to_numpy() for x in reversal_probs_list]

    #print the averages
    print(averages[prediction_type_number].round(1))

    #arrays for where 100s and 0s should go on graph
    tick_100_y = counts_100
    tick_0_y = 100 - counts_0

    #write data as a dataframe
    data = {
        'Index': reversal_probs.columns,
        'Highs': counts_high,
        'Mids': counts_mid,
        'Lows': counts_low
    }
    df = pd.DataFrame(data).set_index('Index')

    fig, ax = plt.subplots(figsize=(7, 5))

    #plot data
    df.plot(kind='bar', stacked=True, ax=ax, width=0.5, color = colors)


    bar_width = 0.5
    xticks = ax.get_xticks()

    #add 100 and 0 dashes
    ax.hlines(tick_100_y, xticks - bar_width/2, xticks + bar_width/2, color='black', linewidth=2, zorder=5)
    ax.hlines(tick_0_y, xticks - bar_width/2, xticks + bar_width/2, color='black', linewidth=2, zorder=5)

    #plot lines for averages
    ax.plot(ax.get_xticks(), averages[0], color='orange', marker='o', linewidth=2.5, label='Original Average')
    ax.plot(ax.get_xticks(), averages[1], color='blue', marker='o', linewidth=2.5, label='Normal Average')
    ax.plot(ax.get_xticks(), averages[2], color='lightblue', marker='o', linewidth=2.5, label='Good Average')

    ax.legend(loc='lower right')

    plt.ylim(0, 100)

    plt.suptitle('Prediction Horizon', fontsize=16, fontweight='bold')
    plt.title('Reversals Predicted Rate by Index', fontsize=12, color='gray')
    plt.xlabel('Window Index')
    plt.ylabel('Reversals Predicted Rate')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

def show_bar_of_prediction_horizon(prediction_type_value = "Normal"):
    #### Interface #### 
    prediction_type = make_prediction_type(value = prediction_type_value)
    ui = widgets.VBox([prediction_type])

    #### Display ####
    out = widgets.interactive_output(
        bar_of_prediction_horizon,
        {
            "prediction_type": prediction_type
        }
    )
    display(ui, out)

show_bar_of_prediction_horizon()

Output()

### Prediction Horizon by Model

In [16]:
def prediction_horizon_by_model(
        prediction_type
    ):

    num_windows = 80 - first_window

    prediction_type_number = prediction_type_list.index(prediction_type)
    pred_reversal = pred_reversal_list[prediction_type_number]

    mean_by_model = pred_reversal.float().mean(dim = 1).detach().cpu().numpy() * 100

    model_names = [f"Model {i}" for i in range(mean_by_model.shape[0])]  
    timesteps = range(first_window, 80) 

    fig, ax = plt.subplots(figsize=(7, 5))

    for i in range(mean_by_model.shape[0]):
        plt.plot(timesteps, mean_by_model[i], label=model_names[i], marker='o')

    ax.legend(loc='lower right')

    plt.ylim(0, 100)
    plt.xlim(first_window, 79)

    plt.suptitle('Prediction Horizon', fontsize=16, fontweight='bold')
    plt.title('Reversals Predicted Rate by Index', fontsize=12, color='gray')
    plt.xlabel('Window Index')
    plt.ylabel('Reversals Predicted Rate')
    plt.tight_layout()
    plt.show()

def show_prediction_horizon_by_model(prediction_type_value = "Normal"):
    #### Interface #### 
    prediction_type = make_prediction_type(value = prediction_type_value)
    ui = widgets.VBox([prediction_type])

    #### Display ####
    out = widgets.interactive_output(
        prediction_horizon_by_model,
        {
            "prediction_type": prediction_type
        }
    )
    display(ui, out)

show_prediction_horizon_by_model()

Output()

### Histogram of Timeseries Subset Summary Stats

In [51]:
#### Define Function####
def hist_of_timeseries_subset_summary_stat(
        window_idx,
        highlight_idx,
        feature_name,
        window_subset,
        summary_statistic,
        show_means,
        show_stream,
        prediction_type,
        highlight_type
    ):

    #vars
    feature_num = feature_list.index(feature_name)
    summary_stat_num = summary_stat_list.index(summary_statistic)
    prediction_type_number = prediction_type_list.index(prediction_type)
    window_subset_start, window_subset_end = window_subset
    bins_list = [[np.linspace(-4, 4, 40), np.linspace(-4, 4, 40), np.linspace(-0.5, 2.5, 30)],
        [np.linspace(-4, 4, 40), np.linspace(-4, 4, 40), np.linspace(-0.5, 2.5, 30)],
        [np.linspace(0, 3, 40), np.linspace(0, 3, 40), np.linspace(0, 1, 30)],
        [np.linspace(-5, 2, 40), np.linspace(-5, 2, 40), np.linspace(-0.5, 2.5, 30)],
        [np.linspace(-2, 5, 40), np.linspace(-2, 5, 40), np.linspace(-0.5, 2.5, 30)],
        [np.linspace(0, 8, 40), np.linspace(0, 8, 40), np.linspace(0, 2, 30)]]

    fig, ax = plt.subplots(figsize = (7, 5))

    #calculate the summary statistics for the selected settings
    #axis = 1 to get the summary statistic over the window subset
    #features_100_1 = calculate_summary_stat(features[0][prediction_type_number], summary_statistic, window_idx, window_subset_start, window_subset_end, feature_num)
    features_high_1 = calculate_summary_stat(features[1][prediction_type_number], summary_statistic, window_idx, window_subset_start, window_subset_end, feature_num, axis = 1)
    features_mid_1 = calculate_summary_stat(features[2][prediction_type_number], summary_statistic, window_idx, window_subset_start, window_subset_end, feature_num, axis = 1)
    features_low_1 = calculate_summary_stat(features[3][prediction_type_number], summary_statistic, window_idx, window_subset_start, window_subset_end, feature_num, axis = 1)
    #features_0_1 = calculate_summary_stat(features[4][prediction_type_number], summary_statistic, window_idx, window_subset_start, window_subset_end, feature_num)

    #t-test comparing between high and low
    t_stat, p_value = stats_module.ttest_ind(features_high_1, features_low_1, equal_var=False)

    bins = bins_list[summary_stat_num][feature_num]

    #plot the stuff
    #plt.hist(features_100_1, bins, alpha=0.5, histtype='step', color = colors[3], label='100')
    plt.hist(features_high_1, bins, alpha=0.5, histtype='step', color = colors[0], label='High')
    plt.hist(features_mid_1, bins, alpha=0.5, histtype='step', color = colors[1], label='Mid')
    plt.hist(features_low_1, bins, alpha=0.5, histtype='step', color = colors[2], label='Low')
    #plt.hist(features_0_1, bins, alpha=0.5, histtype='step', color = colors[4], label='0')

    if(show_stream):
        stream_data = calculate_summary_stat(stream_windows, summary_statistic, window_idx, window_subset_start, window_subset_end, feature_num, axis = 1, count = 500)
        plt.hist(stream_data, bins, alpha=0.5, histtype='step', color = colors[5], label='Stream')

    #vertical line denoting the mean, if we want
    if(show_means):
        plt.axvline(x=features_high_1.mean(), color=colors[0], linestyle='-', linewidth=1)
        plt.axvline(x=features_mid_1.mean(), color=colors[1], linestyle='-', linewidth=1)
        plt.axvline(x=features_low_1.mean(), color=colors[2], linestyle='-', linewidth=1)
        if(show_stream):
            plt.axvline(x=stream_data.mean(), color=colors[5], linestyle='-', linewidth=1)

    if(highlight_type == "Reversal"):
        full_features_summary = calculate_summary_stat(full_features, summary_statistic, window_idx, window_subset_start, window_subset_end, feature_num, axis = 1)
        plt.scatter(full_features_summary[highlight_idx], 0, color = colors[int(colors_vals[prediction_type_number, window_idx - first_window, highlight_idx])], s = 100)
    elif(highlight_type == "Stream"):
        if(show_stream):
            plt.scatter(stream_data[highlight_idx], 0, color = colors[5], s = 100)

    #print p-value
    plt.text(0.02, 0.95, s=f"p-value = {p_value:.4f}", transform=plt.gca().transAxes, fontsize=8, bbox=dict(facecolor='white', alpha=0.5))

    plt.suptitle(f'Histogram of Timeseries {summary_statistic} of {feature_name}', fontsize=16, fontweight='bold')
    plt.title(f'Window Index: {window_idx}     Subset: {window_idx*5 - 500 + window_subset_start}, {window_idx*5 - 500 + window_subset_end}', fontsize=12, color='gray')
    plt.xlabel('Z-Score')
    plt.ylabel('Count')
    plt.legend(title="Reversal Probability", loc='upper right')
    plt.show()

def show_hist_of_timeseries_subset_summary_stat(window_idx_value = 69, prediction_type_value = "Normal", highlight_idx_value = 0,
                                     feature_name_value = "U", summary_stat_value = "Mean", highlight_type_value = "None",
                                     window_subset_value = [0,100], show_means_value = False, show_stream_value = False):

    #### Interface #### 
    window_idx = make_window_idx(value = window_idx_value)
    prediction_type = make_prediction_type(value = prediction_type_value)
    highlight_idx = make_highlight_idx(value = highlight_idx_value)
    feature_name = make_feature_name(value = feature_name_value)
    summary_statistic = make_summary_statistic(value = summary_stat_value)
    highlight_type = make_highlight_type(value = highlight_type_value)
    window_subset = make_window_subset(value = window_subset_value)
    show_means = make_show_means(value = show_means_value)
    show_stream = make_show_stream(value = show_stream_value)

    horizontal = widgets.HBox([feature_name, summary_statistic, prediction_type])
    ui = widgets.VBox([window_idx, window_subset, highlight_idx])
    horizontal2 = widgets.HBox([show_means, show_stream, highlight_type])

    #### Display ####
    out = widgets.interactive_output(
        hist_of_timeseries_subset_summary_stat,
        {
            "window_idx": window_idx,
            "highlight_idx": highlight_idx,
            "feature_name": feature_name,
            "window_subset": window_subset,
            "summary_statistic": summary_statistic,
            "show_means": show_means,
            "show_stream": show_stream,
            "prediction_type": prediction_type,
            "highlight_type": highlight_type,
        }
    )
    display(horizontal, ui, horizontal2, out)

show_hist_of_timeseries_subset_summary_stat(window_idx_value = 69, prediction_type_value = "Normal", highlight_idx_value = 0,
                                     feature_name_value = "U", summary_stat_value = "Mean", highlight_type_value = "None",
                                     window_subset_value = [0,100], show_means_value = False, show_stream_value = False)



Output()

### Time-Series Plot of Index Summary Stats

In [26]:

#function for a bootstrap confidence interval
def bootstrap_ci(data, summary_stat, axis=0, confidence=0.95, n_boot=2000, seed=None):

    #data as array
    data = np.asarray(data) 

    rng = np.random.default_rng(seed)
    n = data.shape[axis]
    
    out_shape = (n_boot,) + tuple(s for i, s in enumerate(data.shape) if i != axis)
    boot_vals = np.empty(out_shape)
    
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        sample = np.take(data, idx, axis=axis)
        if(summary_stat == "Mean"):
            boot_vals[i] = np.mean(sample, axis=axis)
        elif (summary_stat == "Median"):
            boot_vals[i] = np.median(sample, axis = axis)
        elif (summary_stat == "Std"):
            boot_vals[i] = sample.std(axis = axis)
            
    
    lower = np.percentile(boot_vals, (1 - confidence) / 2 * 100, axis=0)
    upper = np.percentile(boot_vals, (1 + confidence) / 2 * 100, axis=0)
    return lower, upper

#### Define Function####
def timeseries_of_index_summary_stat(
        window_idx,
        feature_name,
        summary_statistic,
        prediction_type,
        show_stream
    ):

    #vars
    feature_num = feature_list.index(feature_name)
    summary_stat_num = summary_stat_list.index(summary_statistic)
    prediction_type_number = prediction_type_list.index(prediction_type)
    window_subset_start, window_subset_end = 0, 100
    fig, ax = plt.subplots(figsize = (7, 5))
    relative_start, relative_end = window_idx*5 - 500, window_idx*5 + 100 - 500
    ylims = [[(-3, 3), (-3, 3), (-0.5, 2.5)],
             [(-3, 3), (-3, 3), (-0.5, 2.5)],
             [(0, 3), (0, 3), (0, 1)],
             [(-6, 1), (-6, 1), (-1, 4)],
             [(-1, 6), (-1, 6), (-1, 4)],
             [(0, 10), (0, 10), (0, 4)]]

    #calculate the summary statistics for the selected settings
    #axis = 0 to get the summary statistic over the index
    features_100_2 = calculate_summary_stat(features[0][prediction_type_number], summary_statistic, window_idx, window_subset_start, window_subset_end, feature_num, axis = 0)
    features_high_2 = calculate_summary_stat(features[1][prediction_type_number], summary_statistic, window_idx, window_subset_start, window_subset_end, feature_num, axis = 0)
    features_mid_2 = calculate_summary_stat(features[2][prediction_type_number], summary_statistic, window_idx, window_subset_start, window_subset_end, feature_num, axis = 0)
    features_low_2 = calculate_summary_stat(features[3][prediction_type_number], summary_statistic, window_idx, window_subset_start, window_subset_end, feature_num, axis = 0)
    features_0_2 = calculate_summary_stat(features[4][prediction_type_number], summary_statistic, window_idx, window_subset_start, window_subset_end, feature_num, axis = 0)

    #generate bootstrap confidence intervals
    _100_ci_lower, _100_ci_upper = bootstrap_ci(features[0][prediction_type_number][window_idx - first_window][:, :, feature_num], summary_stat = summary_statistic)
    high_ci_lower, high_ci_upper = bootstrap_ci(features[1][prediction_type_number][window_idx - first_window][:, :, feature_num], summary_stat = summary_statistic)
    mid_ci_lower, mid_ci_upper = bootstrap_ci(features[2][prediction_type_number][window_idx - first_window][:, :, feature_num], summary_stat = summary_statistic)
    low_ci_lower, low_ci_upper = bootstrap_ci(features[3][prediction_type_number][window_idx - first_window][:, :, feature_num], summary_stat = summary_statistic)
    _0_ci_lower, _0_ci_upper = bootstrap_ci(features[4][prediction_type_number][window_idx - first_window][:, :, feature_num], summary_stat = summary_statistic)

    #plot the stuff
    #plt.plot(np.arange(relative_start, relative_end), features_100_2, linestyle='-', marker='o', color=colors[3], label='100')
    plt.plot(np.arange(relative_start, relative_end), features_high_2, linestyle='-', marker='o', color=colors[0], label= f'High (n = {len(features[1][prediction_type_number][window_idx - first_window][:, 0, feature_num])})')
    plt.plot(np.arange(relative_start, relative_end), features_mid_2, linestyle='-', marker='o', color=colors[1], label=f'Mid (n = {len(features[2][prediction_type_number][window_idx - first_window][:, 0, feature_num])})')
    plt.plot(np.arange(relative_start, relative_end), features_low_2, linestyle='-', marker='o', color=colors[2], label=f'Low (n = {len(features[3][prediction_type_number][window_idx - first_window][:, 0, feature_num])})')
    #plt.plot(np.arange(relative_start, relative_end), features_0_2, linestyle='-', marker='o', color=colors[4], label='0')
    
    if(show_stream):
        stream_data = calculate_summary_stat(stream_windows, summary_statistic, window_idx, window_subset_start, window_subset_end, feature_num, axis = 0, count = 1000)
        plt.plot(np.arange(relative_start, relative_end), stream_data, linestyle='-', marker='o', color=colors[5], label= f'High (n = 1000)')

    #plot the confidence intervals if they make sense
    if(summary_statistic in ["Mean", "Median", "Std"]):
        #ax.fill_between(x = np.arange(relative_start, relative_end), y1 = _100_ci_lower, y2 = _100_ci_upper, color = colors[3], alpha = 0.25)
        ax.fill_between(x = np.arange(relative_start, relative_end), y1 = high_ci_lower, y2 = high_ci_upper, color = colors[0], alpha = 0.25)
        ax.fill_between(x = np.arange(relative_start, relative_end), y1 = mid_ci_lower, y2 = mid_ci_upper, color = colors[1], alpha = 0.25)
        ax.fill_between(x = np.arange(relative_start, relative_end), y1 = low_ci_lower, y2 = low_ci_upper, color = colors[2], alpha = 0.25)
        #ax.fill_between(x = np.arange(relative_start, relative_end), y1 = _0_ci_lower, y2 = _0_ci_upper, color = colors[4], alpha = 0.25)

    plt.ylim(ylims[summary_stat_num][feature_num])

    plt.suptitle(f'Time-series Plot of Index {summary_statistic} of {feature_name} ', fontsize=16, fontweight='bold')
    plt.title(f'Window Index: {window_idx}', fontsize=12, color='gray')
    plt.xlabel('Index')
    plt.ylabel('Z-Score')
    plt.legend(title="Reversal Probability", loc='upper right')
    plt.show()


def show_hist_of_timeseries_subset_summary_stat(window_idx_value = 69, prediction_type_value = "Normal",
                                     feature_name_value = "U", summary_stat_value = "Mean", show_stream_value = False):

    #### Interface #### 
    window_idx = make_window_idx(value = window_idx_value)
    prediction_type = make_prediction_type(value = prediction_type_value)
    feature_name = make_feature_name(value = feature_name_value)
    summary_statistic = make_summary_statistic(value = summary_stat_value)
    show_stream = make_show_stream(value = show_stream_value)

    horizontal = widgets.HBox([feature_name, summary_statistic, prediction_type])
    ui = widgets.VBox([window_idx, show_stream])

    #### Display ####
    out = widgets.interactive_output(
        timeseries_of_index_summary_stat,
        {
            "window_idx": window_idx,
            "feature_name": feature_name,
            "summary_statistic": summary_statistic,
            "show_stream": show_stream,
            "prediction_type": prediction_type
        }
    )
    display(horizontal, ui, out)


show_hist_of_timeseries_subset_summary_stat(window_idx_value = 69, prediction_type_value = "Normal",
        feature_name_value = "U", summary_stat_value = "Mean", show_stream_value = False)


Output()

### Histogram by Index

In [ ]:
#### Define Function####
def hist_of_index(
        window_idx,
        point_idx,
        highlight_idx,
        feature_name,
        show_means,
        show_stream,
        prediction_type,
        highlight_type
    ):

    #vars
    feature_num = feature_list.index(feature_name)
    prediction_type_number = prediction_type_list.index(prediction_type)
    fig, ax = plt.subplots(figsize = (7, 5))
    bins_list = [np.linspace(-4, 4, 30), np.linspace(-4, 4, 30), np.linspace(-0.5, 2.5, 30)]

    #get the relevant data
    features_100_3 = features[0][prediction_type_number][window_idx - first_window][:, point_idx, feature_num]
    features_high_3 = features[1][prediction_type_number][window_idx - first_window][:, point_idx, feature_num]
    features_mid_3 = features[2][prediction_type_number][window_idx - first_window][:, point_idx, feature_num]
    features_low_3 = features[3][prediction_type_number][window_idx - first_window][:, point_idx, feature_num]
    features_0_3 = features[4][prediction_type_number][window_idx - first_window][:, point_idx, feature_num]

    
    bins = bins_list[feature_num]

    #plot the stuff
    #plt.hist(features_100_3, bins, alpha=0.5, histtype='step', color = colors[3], label='100')
    plt.hist(features_high_3, bins, alpha=0.5, histtype='step', color = colors[0], label='High')
    plt.hist(features_mid_3, bins, alpha=0.5, histtype='step', color = colors[1], label='Mid')
    plt.hist(features_low_3, bins, alpha=0.5, histtype='step', color = colors[2], label='Low')
    #plt.hist(features_0_3 bins, alpha=0.5, histtype='step', color = colors[4], label='0')

    if(show_stream):
        stream_data = stream_windows[0:499, point_idx, feature_num]
        plt.hist(stream_data, bins, alpha=0.5, histtype='step', color = colors[5], label='Stream')

    if(show_means):
        plt.axvline(x=features_high_3.mean(), color=colors[0], linestyle='-', linewidth=1)
        plt.axvline(x=features_mid_3.mean(), color=colors[1], linestyle='-', linewidth=1)
        plt.axvline(x=features_low_3.mean(), color=colors[2], linestyle='-', linewidth=1)
        if(show_stream):
            plt.axvline(x=stream_data.mean(), color=colors[5], linestyle='-', linewidth=1)

    if(highlight_type == "Reversal"):
        full_features_relevant = full_features[highlight_idx, window_idx - first_window, point_idx, feature_num]
        plt.scatter(full_features_relevant, 0, color = colors[int(colors_vals[prediction_type_number, window_idx - first_window, highlight_idx])], s = 100)
    elif(highlight_type == "Stream"):
        if(show_stream):
            plt.scatter(stream_data[highlight_idx], 0, color = colors[5], s = 100)

    plt.ylim(0, 100)

    plt.suptitle(f'Histogram of {feature_name}', fontsize=16, fontweight='bold')
    plt.title(f'Window Index: {window_idx}      Point Index: {window_idx*5 - 500 + point_idx}', fontsize=12, color='gray')
    plt.xlabel('Z Score')
    plt.ylabel('Count')
    plt.legend(title="Reversal Probability", loc='upper right')
    plt.show()

def show_hist_of_index(window_idx_value = 69, point_idx_value = 0, prediction_type_value = "Normal", highlight_idx_value = 0,
                                     feature_name_value = "U", highlight_type_value = "None",
                                     show_means_value = False, show_stream_value = False):

    #### Interface #### 
    window_idx = make_window_idx(value = window_idx_value)
    point_idx = make_point_idx_100(value = point_idx_value)
    prediction_type = make_prediction_type(value = prediction_type_value)
    highlight_idx = make_highlight_idx(value = highlight_idx_value)
    feature_name = make_feature_name(value = feature_name_value)
    highlight_type = make_highlight_type(value = highlight_type_value)
    show_means = make_show_means(value = show_means_value)
    show_stream = make_show_stream(value = show_stream_value)

    horizontal = widgets.HBox([feature_name, prediction_type])
    ui = widgets.VBox([window_idx, point_idx, highlight_idx])
    horizontal2 = widgets.HBox([show_means, show_stream, highlight_type])

    #### Display ####
    out = widgets.interactive_output(
        hist_of_index,
        {
            "window_idx": window_idx,
            "point_idx": point_idx,
            "highlight_idx": highlight_idx,
            "feature_name": feature_name,
            "show_means": show_means,
            "show_stream": show_stream,
            "prediction_type": prediction_type,
            "highlight_type": highlight_type,
        }
    )
    display(horizontal, ui, horizontal2, out)

show_hist_of_index(window_idx_value = 69, point_idx_value = 0, prediction_type_value = "Normal", highlight_idx_value = 0,
                                     feature_name_value = "U", highlight_type_value = "None",
                                     show_means_value = False, show_stream_value = False)



Output()

### Line Graph of Windows

In [134]:
#### Define Function####
def line_graph_of_features(
        window_idx,
        highlight_idx,
        count,
        feature_name,
        prediction_type,
        highlight_type,
        show_high,
        show_low,
        show_stream
    ):

    #vars
    feature_num = feature_list.index(feature_name)
    prediction_type_number = prediction_type_list.index(prediction_type)
    fig, ax = plt.subplots(figsize = (7, 5))
    relative_start, relative_end = window_idx*5 - 500, window_idx*5 + 100 - 500
    ylims = [(-4, 4), (-4, 4), (-1, 3)]

    if(highlight_type == "None"):
        alpha_val = 1
    else:
        alpha_val = 0.45

    #get the relevant data
    features_100_3 = features[0][prediction_type_number][window_idx - first_window][:, :, feature_num]
    features_high_3 = features[1][prediction_type_number][window_idx - first_window][:, :, feature_num]
    features_mid_3 = features[2][prediction_type_number][window_idx - first_window][:, :, feature_num]
    features_low_3 = features[3][prediction_type_number][window_idx - first_window][:, :, feature_num]
    features_0_3 = features[4][prediction_type_number][window_idx - first_window][:, :, feature_num]


    if(show_stream):
        stream_data = stream_windows[:, :, feature_num]
        for i in range(min(count, len(stream_data))):
            plt.plot(np.arange(relative_start, relative_end), stream_data[i], color = colors[5], label='Stream' if i == 0 else None, alpha = alpha_val)

    if(show_high):
        for i in range(min(count, len(features_high_3))):
            plt.plot(np.arange(relative_start, relative_end), features_high_3[i], color = colors[0], label='High' if i == 0 else None, alpha = alpha_val)

    if(show_low):
        for i in range(min(count, len(features_low_3))):
            plt.plot(np.arange(relative_start, relative_end), features_low_3[i], color = colors[2], label='Low' if i == 0 else None, alpha = alpha_val)

    if(highlight_type == "Reversal"):
        plt.plot(np.arange(relative_start, relative_end), full_features[highlight_idx, window_idx - first_window, :, feature_num], color = colors[int(colors_vals[prediction_type_number, window_idx - first_window, highlight_idx])], lw = 3)
    if(highlight_type == "Stream"):
        if(show_stream):
            plt.plot(np.arange(relative_start, relative_end), stream_data[highlight_idx], color = colors[5], lw = 3)


    plt.ylim(ylims[feature_num])
    plt.margins(x=0)

    plt.suptitle(f'Line Graph of {feature_name}', fontsize=16, fontweight='bold')
    plt.title(f'Window Index: {window_idx}', fontsize=12, color='gray')
    plt.xlabel('Index')
    plt.ylabel('Z Score')
    plt.legend(title="Reversal Probability", loc='upper left')
    plt.show()

def show_line_graph_of_features(window_idx_value = 69, highlight_idx_value = 0, count_value = 50, 
                                     feature_name_value = "U",  prediction_type_value = "Normal", highlight_type_value = "None", 
                                     show_high_value = True, show_low_value = True, show_stream_value = False):

    #### Interface #### 
    window_idx = make_window_idx(value = window_idx_value)
    highlight_idx = make_highlight_idx(value = highlight_idx_value)
    count = make_count(value = count_value)
    feature_name = make_feature_name(value = feature_name_value)
    prediction_type = make_prediction_type(value = prediction_type_value)
    highlight_type = make_highlight_type(value = highlight_type_value)
    show_high = make_show_high(value = show_high_value)
    show_low = make_show_low(value = show_low_value)
    show_stream = make_show_stream(value = show_stream_value)

    horizontal = widgets.HBox([feature_name, prediction_type, highlight_type])
    ui = widgets.VBox([window_idx, highlight_idx, count])
    horizontal2 = widgets.HBox([show_high, show_low, show_stream])

    #### Display ####
    out = widgets.interactive_output(
        line_graph_of_features,
        {
            "window_idx": window_idx,
            "highlight_idx": highlight_idx,
            "count": count,
            "feature_name": feature_name,
            "prediction_type": prediction_type,
            "highlight_type": highlight_type,
            "show_high": show_high,
            "show_low": show_low,
            "show_stream": show_stream
        }
    )
    display(horizontal, ui, horizontal2, out)

show_line_graph_of_features(window_idx_value = 69, highlight_idx_value = 0, count_value = 50,
            feature_name_value = "U", prediction_type_value = "Normal", highlight_type_value = "None",
            show_high_value = True, show_low_value = True, show_stream_value = False)



Output()

### Bar Graph of Prediction Rate by Model

In [18]:
#### Define Function####
def bar_of_pred_reversal_by_model(
        window_idx,
        prediction_type,
        target_prob,
        aggregate
    ):

    #vars
    prediction_type_number = prediction_type_list.index(prediction_type)
    pred_reversal = pred_reversal_list[prediction_type_number]
    num_windows = 80 - first_window

    count_by_model = np.zeros(len(model_paths))

    if(aggregate):
        for i in range(num_timeseries):
            for j in range(num_windows):
                if(pred_reversal[:, i, j].float().mean().detach().cpu().numpy() * 100 == target_prob):
                    preds = np.asarray(pred_reversal[:, i, j])
                    count_by_model += preds
        y_lim = max(max(count_by_model) + 20, 120)

    else:
        for i in range(num_timeseries):
            if(pred_reversal[:, i, window_idx - first_window].float().mean().detach().cpu().numpy() * 100 == target_prob):
                preds = np.asarray(pred_reversal[:, i, window_idx - first_window])
                count_by_model += preds
        y_lim = 100


    fig, ax = plt.subplots(figsize=(7, 5))

    plt.bar(np.arange(len(count_by_model)), count_by_model, width=0.5)

    plt.ylim(0, y_lim)

    plt.suptitle('Model Prediction Rates', fontsize=16, fontweight='bold')
    plt.xlabel('Model Number')
    plt.ylabel('Reversals Predicted Count')
    plt.tight_layout()
    plt.show()

def show_bar_of_pred_reversal_by_model(window_idx_value = 69, prediction_type_value = "Normal", target_prob_value = 0, aggregate_value = False):

    #### Interface #### 
    window_idx = make_window_idx(value = window_idx_value)
    prediction_type = make_prediction_type(value = prediction_type_value)
    target_prob = make_target_prob(value = target_prob_value)
    aggregate = make_aggregate(value = aggregate_value)

    horizontal = widgets.HBox([prediction_type, aggregate])
    ui = widgets.VBox([window_idx, target_prob])

    #### Display ####
    out = widgets.interactive_output(
        bar_of_pred_reversal_by_model,
        {
            "prediction_type": prediction_type,
            "window_idx": window_idx,
            "target_prob": target_prob,
            "aggregate": aggregate
        }
    )   
    display(horizontal, ui, out)

show_bar_of_pred_reversal_by_model(window_idx_value = 69, prediction_type_value = "Normal", target_prob_value = 0, aggregate_value = False)



Output()

### Histogram of Residuals by Index

In [43]:
#### Define Function####
def hist_of_residuals(
        window_idx,
        point_idx,
        highlight_idx,
        model_idx,
        feature_name,
        show_means,
        show_stream,
        combine_models,
        prediction_type,
        highlight_type
    ):

    #vars
    feature_num = feature_list.index(feature_name)
    prediction_type_number = prediction_type_list.index(prediction_type)
    fig, ax = plt.subplots(figsize = (7, 5))
    bins_list = [np.linspace(-3, 3, 40), np.linspace(-3, 3, 40), np.linspace(-3, 3, 40)]

    if(combine_models):
        #get the relevant data
        resids_true_temp = [tensor[:, point_idx, feature_num] for tensor in residuals[0][prediction_type_number][window_idx - first_window]]
        resids_true = torch.cat(resids_true_temp)

        resids_false_temp = [tensor[:, point_idx, feature_num] for tensor in residuals[1][prediction_type_number][window_idx - first_window]]
        resids_false = torch.cat(resids_false_temp)

        resids_stream_temp = stream_residuals[:, 0:499, point_idx, feature_num]
        resids_stream = resids_stream_temp.flatten()

        plt.ylim(0, 1000)
    else:
        resids_true = residuals[0][prediction_type_number][window_idx - first_window][model_idx][:, point_idx, feature_num]
        resids_false = residuals[1][prediction_type_number][window_idx - first_window][model_idx][:, point_idx, feature_num]
        resids_stream = stream_residuals[model_idx, 0:499, point_idx, feature_num]
        plt.ylim(0, 200)

    bins = bins_list[feature_num]

    #plot the stuff
    plt.hist(resids_true, bins, alpha=0.5, histtype='step', color = colors[0], label='True')
    plt.hist(resids_false, bins, alpha=0.5, histtype='step', color = colors[2], label='False')

    if(show_stream):
        plt.hist(resids_stream, bins, alpha=0.5, histtype='step', color = colors[5], label='Stream')
        if(highlight_type == "Stream"):
            if(combine_models):
                plt.scatter(stream_residuals[:, highlight_idx, point_idx, feature_num], np.zeros(10), color = colors[5], s = 20)
            else:
                plt.scatter(stream_residuals[model_idx, highlight_idx, point_idx, feature_num], 0, color = colors[5], s = 100)

    if(show_means):
        plt.axvline(x=resids_true.mean(), color=colors[0], linestyle='-', linewidth=1)
        plt.axvline(x=resids_false.mean(), color=colors[2], linestyle='-', linewidth=1)

    if(highlight_type == "Reversal"):
        if(combine_models):
            full_resids_relevant = full_residuals[:, highlight_idx, window_idx - first_window, point_idx, feature_num]
            color_idx = colors_vals_by_mod[prediction_type_number, :, window_idx - first_window, highlight_idx]
            plt.scatter(full_resids_relevant, np.zeros(10), color =[colors[int(i)] for i in color_idx], s = 20)
        else:
            full_resids_relevant = full_residuals[model_idx, highlight_idx, window_idx - first_window, point_idx, feature_num]
            plt.scatter(full_resids_relevant, 0, color = colors[int(colors_vals_by_mod[prediction_type_number, model_idx, window_idx - first_window, highlight_idx])], s = 100)
            

    plt.suptitle(f'Histogram of Residuals for {feature_name}', fontsize=16, fontweight='bold')
    plt.title(f'Window Index: {window_idx}      Point Index: {window_idx*5 - 400 + point_idx}', fontsize=12, color='gray')
    plt.xlabel('Residual')
    plt.ylabel('Count')
    plt.legend(title="Reversal Prediction", loc='upper right')
    plt.show()

def show_hist_of_residuals(window_idx_value = 69, point_idx_value = 0, highlight_idx_value = 0, model_idx_value = 0,
                                     feature_name_value = "U", combine_models_value = True, prediction_type_value = "Normal", highlight_type_value = "None",
                                     show_means_value = False, show_stream_value = False):

    #### Interface #### 
    window_idx = make_window_idx(value = window_idx_value)
    point_idx = make_point_idx_50(value = point_idx_value)
    highlight_idx = make_highlight_idx(value = highlight_idx_value)
    model_idx = make_model_idx(value = model_idx_value)
    feature_name = make_feature_name(value = feature_name_value)
    combine_models = make_combine_models(value = combine_models_value)
    prediction_type = make_prediction_type(value = prediction_type_value)
    highlight_type = make_highlight_type(value = highlight_type_value)
    show_means = make_show_means(value = show_means_value)
    show_stream = make_show_stream(value = show_stream_value)

    horizontal = widgets.HBox([feature_name, prediction_type, highlight_type])
    ui = widgets.VBox([window_idx, point_idx, highlight_idx, model_idx])
    horizontal2 = widgets.HBox([show_means, show_stream, combine_models])

    def toggle_model_idx(change):
        if change["new"]:  # combine_models is True
            model_idx.layout.display = "none"
        else:
            model_idx.layout.display = ""  # or "flex", depending on your layout

    # Set initial visibility based on the starting value
    toggle_model_idx({"new": combine_models.value})

    # Watch for future changes
    combine_models.observe(toggle_model_idx, names="value")

    #### Display ####
    out = widgets.interactive_output(
        hist_of_residuals,
        {
            "window_idx": window_idx,
            "point_idx": point_idx,
            "highlight_idx": highlight_idx,
            "model_idx": model_idx,
            "feature_name": feature_name,
            "combine_models": combine_models,
            "prediction_type": prediction_type,
            "highlight_type": highlight_type,
            "show_means": show_means,
            "show_stream": show_stream
        }
    )
    display(horizontal, ui, horizontal2, out)

show_hist_of_residuals(window_idx_value = 69, point_idx_value = 0, highlight_idx_value = 0, model_idx_value = 0,
                                     feature_name_value = "U", combine_models_value = True, prediction_type_value = "Normal", highlight_type_value = "None",
                                     show_means_value = False, show_stream_value = False)


Output()

### Residual Scatterplot by Index

In [75]:
#### Define Function####
def scatterplot_of_resids(
        window_idx,
        point_idx,
        highlight_idx,
        model_idx,
        feature_name,
        prediction_type,
        highlight_type,
        show_lines,
        show_stream
    ):

    #vars
    feature_num = feature_list.index(feature_name)
    prediction_type_number = prediction_type_list.index(prediction_type)
    fig, ax = plt.subplots(figsize = (7, 5))
    relative_start, relative_end = window_idx*5 - 500 + 100, window_idx*5 + 50 - 500 + 100
    ylims = [(-4, 4), (-4, 4), (-4, 4)]
    xlims = [(-4, 4), (-4, 4), (-4, 4)]

    #data
    resids = np.asarray(full_residuals[model_idx, :, window_idx - first_window, point_idx, feature_num])
    y_values = np.asarray(full_y[:, window_idx - first_window, point_idx, feature_num])

    #trues and falses
    predictions = np.asarray(pred_reversal_list[prediction_type_number][model_idx, :, window_idx - first_window])

    if(show_stream):
        plt.scatter(stream_y[0:499, point_idx, feature_num], stream_residuals[model_idx, 0:499, point_idx, feature_num], marker='o', s = 14, color=colors[5], label='Stream')
        
    #plot the stuff
    plt.scatter(y_values[predictions], resids[predictions], marker='o', s = 14, color=colors[0], label='True')
    plt.scatter(y_values[~predictions], resids[~predictions], marker='o', s = 14, color=colors[2], label='False')

    if(highlight_type == "Reversal"):
        plt.scatter(y_values[highlight_idx], resids[highlight_idx], marker='o', s = 70, edgecolors='black', color=colors[int(colors_vals_by_mod[prediction_type_number, model_idx, window_idx - first_window, highlight_idx])])

    if((highlight_type == "Stream") & show_stream):
        plt.scatter(stream_y[highlight_idx, point_idx, feature_num], stream_residuals[model_idx, highlight_idx, point_idx, feature_num], marker='o', s = 70, edgecolors='black', color = "green")

        
    #lines
    if(show_lines):
        plt.axline((-2, 0), slope=1, color='black', linestyle='-')
        plt.axline((0, -2), slope=1, color='black', linestyle='-')
    
    plt.ylim(ylims[feature_num])
    plt.xlim(xlims[feature_num])

    plt.suptitle(f'Residual Plot of {feature_name} ', fontsize=16, fontweight='bold')
    plt.title(f'Window Index: {window_idx}      Point Index: {window_idx*5 - 400 + point_idx}', fontsize=12, color='gray')
    plt.xlabel('Y Value')
    plt.ylabel('Residual')
    plt.legend(title="Reversal Prediction", loc='upper right')
    plt.show()

def show_scatterplot_of_resids(window_idx_value = 69, point_idx_value = 0, highlight_idx_value = 0,
                     model_idx_value=0, feature_name_value = "b_plus", prediction_type_value = "Normal", highlight_type_value = "None",
                     show_lines_value = False, show_stream_value = False):

    window_idx = make_window_idx(value = window_idx_value)
    point_idx = make_point_idx_50(value = point_idx_value)
    highlight_idx = make_highlight_idx(value = highlight_idx_value)
    model_idx = make_model_idx(value = model_idx_value)
    feature_name = make_feature_name(value = feature_name_value)
    prediction_type = make_prediction_type(value = prediction_type_value)
    highlight_type = make_highlight_type(value = highlight_type_value)
    show_lines = make_show_lines(value = show_lines_value)
    show_stream = make_show_stream(value = show_stream_value)

    horizontal = widgets.HBox([feature_name, prediction_type, highlight_type])
    ui = widgets.VBox([window_idx, point_idx, highlight_idx, model_idx])
    horizontal2 = widgets.HBox([show_stream, show_lines])

    #### Display ####
    out = widgets.interactive_output(
        scatterplot_of_resids,
        {
            "window_idx": window_idx,
            "point_idx": point_idx,
            "highlight_idx": highlight_idx,
            "model_idx": model_idx,
            "feature_name": feature_name,
            "prediction_type": prediction_type,
            "highlight_type": highlight_type,
            "show_lines": show_lines,
            "show_stream": show_stream
        }
    )
    display(horizontal, ui, horizontal2, out)

show_scatterplot_of_resids(window_idx_value = 69, point_idx_value = 0, highlight_idx_value = 0,
                     model_idx_value=0, feature_name_value = "b_plus", prediction_type_value = "Normal", highlight_type_value = "None",
                     show_lines_value = False, show_stream_value = False)

Output()

### Full Line Graph

In [ ]:
#### Define Function####
def line_graph_of_features(
        window_idx,
        highlight_idx,
        count,
        feature_name,
        prediction_type,
        highlight_type,
        show_high,
        show_low,
        show_stream,
        combine_count
    ):

    full_data = torch.cat((full_features, full_y), dim=2)

    #vars
    feature_num = feature_list.index(feature_name)
    prediction_type_number = prediction_type_list.index(prediction_type)
    reversal_probs = reversal_probs_list[prediction_type_number]

    fig, ax = plt.subplots(figsize = (10.5, 5))
    relative_start, relative_end = window_idx*5 - 500, window_idx*5 + 150 - 500
    ylims = [(-4, 4), (-4, 4), (-3, 3)]
    count_high, count_low = 0, 0
    high_idx, low_idx = 0, 0

    if(highlight_type == "None"):
        alpha_val = 1
    else:
        alpha_val = 0.45

    if(show_stream):
        stream_data = torch.cat((stream_windows[:, :, feature_num], stream_y[:, :, feature_num]), dim=1)
        for i in range(min(count, len(stream_data))):
            plt.plot(np.arange(relative_start, relative_end), stream_data[i], color = colors[5], label='Stream' if i == 0 else None, alpha = alpha_val)

    if(combine_count):
        for i in range(count):
            if(show_high):
                if(float(reversal_probs.iloc[i, window_idx - first_window]) >= 80):
                    plt.plot(np.arange(relative_start, relative_end), full_data[i, window_idx - first_window, :, feature_num], color = colors[0], label='High' if count_high == 0 else None, alpha = alpha_val)
                    count_high = count_high + 1
            if(show_low):
                if(float(reversal_probs.iloc[i, window_idx - first_window]) <= 20):
                    plt.plot(np.arange(relative_start, relative_end), full_data[i, window_idx - first_window, :, feature_num], color = colors[2], label='Low' if count_low == 0 else None, alpha = alpha_val)
                    count_low = count_low + 1
    else:
        if(show_high):
            while((count_high < count) & (high_idx < full_data.shape[0])):
                if(float(reversal_probs.iloc[high_idx, window_idx - first_window]) >= 80):
                    plt.plot(np.arange(relative_start, relative_end), full_data[high_idx, window_idx - first_window, :, feature_num], color = colors[0], label='High' if count_high == 0 else None, alpha = alpha_val)
                    count_high = count_high + 1
                high_idx = high_idx + 1
        if(show_low):
            while((count_low < count) & (low_idx < full_data.shape[0])):
                if(float(reversal_probs.iloc[low_idx, window_idx - first_window]) <= 20):
                    plt.plot(np.arange(relative_start, relative_end), full_data[low_idx, window_idx - first_window, :, feature_num], color = colors[2], label='Low' if count_low == 0 else None, alpha = alpha_val)
                    count_low = count_low + 1
                low_idx = low_idx + 1


    if(highlight_type == "Reversal"):
        plt.plot(np.arange(relative_start, relative_end), full_data[highlight_idx, window_idx - first_window, :, feature_num], color = colors[int(colors_vals[prediction_type_number, window_idx - first_window, highlight_idx])], lw = 3)
    if(highlight_type == "Stream"):
        if(show_stream):
            plt.plot(np.arange(relative_start, relative_end), stream_data[highlight_idx], color = colors[5], lw = 3)


    plt.ylim(ylims[feature_num])
    plt.margins(x=0)
    plt.axvline(x=relative_start + 100, color="black", linestyle='-', linewidth=1)

    plt.suptitle(f'Line Graph of {feature_name}', fontsize=16, fontweight='bold')
    plt.title(f'Window Index: {window_idx}', fontsize=12, color='gray')
    plt.xlabel('Index')
    plt.ylabel('Z Score')
    plt.legend(title="Reversal Probability", loc='upper left')
    plt.show()

def show_line_graph_of_features(window_idx_value = 69, highlight_idx_value = 0, count_value = 50, 
                                     feature_name_value = "U",  prediction_type_value = "Normal", highlight_type_value = "None", 
                                     show_high_value = True, show_low_value = True, show_stream_value = False, combine_count_value = True):

    #### Interface #### 
    window_idx = make_window_idx(value = window_idx_value)
    highlight_idx = make_highlight_idx(value = highlight_idx_value)
    count = make_count(value = count_value)
    feature_name = make_feature_name(value = feature_name_value)
    prediction_type = make_prediction_type(value = prediction_type_value)
    highlight_type = make_highlight_type(value = highlight_type_value)
    show_high = make_show_high(value = show_high_value)
    show_low = make_show_low(value = show_low_value)
    show_stream = make_show_stream(value = show_stream_value)
    combine_count = make_combine_count(value  = combine_count_value)

    horizontal = widgets.HBox([feature_name, prediction_type, highlight_type])
    ui = widgets.VBox([window_idx, highlight_idx, count])
    horizontal2 = widgets.HBox([show_high, show_low, show_stream, combine_count])

    #### Display ####
    out = widgets.interactive_output(
        line_graph_of_features,
        {
            "window_idx": window_idx,
            "highlight_idx": highlight_idx,
            "count": count,
            "feature_name": feature_name,
            "prediction_type": prediction_type,
            "highlight_type": highlight_type,
            "show_high": show_high,
            "show_low": show_low,
            "show_stream": show_stream,
            "combine_count": combine_count
        }
    )
    display(horizontal, ui, horizontal2, out)

show_line_graph_of_features(window_idx_value = 69, highlight_idx_value = 0, count_value = 50,
            feature_name_value = "U", prediction_type_value = "Normal", highlight_type_value = "None",
            show_high_value = True, show_low_value = True, show_stream_value = False, combine_count_value = True)



Output()

# Highlights

### Prediction Horizon


- Window Indices 71-73 the important ones
- Models in agreement most of the time (a lot of 100s and 0s)

In [18]:
show_bar_of_prediction_horizon()

Output()

### Histogram of Mean of U Across Windows

- Mean U across the window tends to be higher for timeseries with high prediction rates.
    - More prevalent the later in window
- Stream is about in the middle, but closer to the low prediction rates.
    - Shows one reason why the model is not predicting a reversal

In [19]:

show_hist_of_timeseries_subset_summary_stat(window_idx_value = 72, prediction_type_value = "Normal", highlight_idx_value = 0,
                                     feature_name_value = "U", summary_stat_value = "Mean", highlight_type_value = "None",
                                     window_subset_value = [85,100], show_means_value = False, show_stream_value = False)

Output()

### Time-Series Plot of Mean U

- Dip in Mean of U for timeseries with high prediction rates around -60 to -55
- We can see in histogram at point indices 75-99
- Lows are roughly in line with stream
    - Again pointing to why the ensemble isn't able to predict a reversal

In [ ]:
show_hist_of_timeseries_subset_summary_stat(window_idx_value =72, prediction_type_value = "Good",
        feature_name_value = "U", summary_stat_value = "Mean", show_stream_value = True)

show_hist_of_index(window_idx_value = 72, point_idx_value = 75, prediction_type_value = "Good", highlight_idx_value = 0,
                                     feature_name_value = "U", highlight_type_value = "None",
                                     show_means_value = False, show_stream_value = False)

Output()

Output()

### Time-Series Plot of Std b_plus

- All stds above stream
- Big jump in std of b_plus for highs at -40
- b_plus gets super bimodal near -35
- Idea that b_plus (and b_e) is in phase before reversals

In [50]:
show_hist_of_timeseries_subset_summary_stat(window_idx_value = 73, prediction_type_value = "Good",
        feature_name_value = "b_plus", summary_stat_value = "Std", show_stream_value = True)

show_hist_of_index(window_idx_value = 73, point_idx_value = 75, prediction_type_value = "Good", highlight_idx_value = 0,
                                     feature_name_value = "b_plus", highlight_type_value = "None",
                                     show_means_value = False, show_stream_value = False)

Output()

Output()

### Histogram of b_e by Index

- Point Indices 79-86
    - Highs bimodal, lows unimodal
        - Lows again more in the distribution of stream
- Point Indices 95-99
    - Both bimodal
        - Highs more so
        - Maybe shows an area the model could do better
- This shows that the phase of b_e matters
    - Phases tend to be more in sync for reversals earlier than 40 timesteps before reversal

In [ ]:
show_hist_of_index(window_idx_value = 72, point_idx_value = 81, prediction_type_value = "Normal", highlight_idx_value = 0,
                                     feature_name_value = "b_e", highlight_type_value = "None",
                                     show_means_value = False, show_stream_value = False)

show_line_graph_of_features(window_idx_value = 72, highlight_idx_value = 0, count_value = 50,
            feature_name_value = "b_e", prediction_type_value = "Normal", highlight_type_value = "None",
            show_true_value = True, show_false_value = True, show_stream_value = False)

Output()

Output()

### Residuals

- Underpredicting extremes for b_plus
- Around point index -21
    - False predictions have more extreme b_plus values and larger residuals
    - Easier to predict when b_plus is around 0
- Errors start early
    - Crucial points are around -20 to -25 for b_e and b_plus
- Can't seem to predict values more extreme than +- 2

In [46]:
show_scatterplot_of_resids(window_idx_value = 72, point_idx_value = 19, highlight_idx_value = 0,
                     model_idx_value=0, feature_name_value = "b_plus", prediction_type_value = "Normal", highlight_type_value = "None",
                     show_lines_value = False, show_stream_value = False)

show_hist_of_residuals(window_idx_value = 72, point_idx_value = 15, highlight_idx_value = 0, model_idx_value = 0,
                                     feature_name_value = "b_e", combine_models_value = True, prediction_type_value = "Normal", highlight_type_value = "None",
                                     show_means_value = False, show_stream_value = False)

Output()

Output()